# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [2]:
import pandas as pd

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}


def clean(path):
    df = pd.read_csv(path)
    print("прочитали:", len(df))

    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    assert df["amount"].isna().sum() == 0
    print("суммы в числа:", len(df))

    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    assert df["city"].isna().sum() == 0
    print("города через словарь:", len(df), "| городов:", df["city"].nunique())

    df = df.drop_duplicates(subset="order_id")
    assert df["order_id"].is_unique
    print("без дублей:", len(df))

    df["order_date"] = pd.to_datetime(df["order_date"])
    print("даты:", len(df))

    df = df[df["customer_id"] != "TEST"].reset_index(drop=True)
    print("без TEST:", len(df))
    return df


df = clean("orders.csv")

прочитали: 5463
суммы в числа: 5463
города через словарь: 5463 | городов: 6
без дублей: 5283
даты: 5283
без TEST: 5280


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [3]:
paid = df["status"] == "оплачен"
sep = df["order_date"].dt.month == 9
octo = df["order_date"].dt.month == 10

s = df.loc[paid & sep, "amount"].sum()
o = df.loc[paid & octo, "amount"].sum()
print(f"Всего: сентябрь {s:,.0f} ₸, октябрь {o:,.0f} ₸, изменение {(o - s) / s * 100:+.1f}%")
print()

for c in sorted(df["city"].unique()):
    m = paid & (df["city"] == c)
    cs = df.loc[m & sep, "amount"].sum()
    co = df.loc[m & octo, "amount"].sum()
    print(f"{c:<10} {cs:>12,.0f} {co:>12,.0f} {(co - cs) / cs * 100:>+7.1f}%")

Всего: сентябрь 35,243,410 ₸, октябрь 34,419,520 ₸, изменение -2.3%

Актобе        2,576,960    2,540,420    -1.4%
Алматы       14,501,070   14,643,020    +1.0%
Астана        9,034,220    8,850,530    -2.0%
Атырау          923,990      969,860    +5.0%
Караганда     3,648,520    3,371,680    -7.6%
Шымкент       4,558,650    4,044,010   -11.3%


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [4]:
def change(d):
    p = d["status"] == "оплачен"
    s = d.loc[p & (d["order_date"] < "2026-10"), "amount"].sum()
    o = d.loc[p & (d["order_date"] >= "2026-10"), "amount"].sum()
    return (o - s) / s * 100


raw = pd.read_csv("orders.csv")
raw["amount"] = pd.to_numeric(raw["amount"].str.replace(" ", ""))
print(f"исходные данные: {len(raw)} строк, {change(raw):+.1f}%")

step = raw.copy()
step["city"] = step["city"].str.strip().str.lower().map(CITY)
print(f"города: убрано {len(raw) - len(step)}, {change(step):+.1f}%")

n = len(step)
step = step.drop_duplicates(subset="order_id")
print(f"дубли: убрано {n - len(step)}, {change(step):+.1f}%")

n = len(step)
step = step[step["customer_id"] != "TEST"]
print(f"TEST: убрано {n - len(step)}, {change(step):+.1f}%")

only_test = raw[raw["customer_id"] != "TEST"]
print(f"если убрать только TEST: {change(only_test):+.1f}%")

исходные данные: 5463 строк, -22.7%
города: убрано 0, -22.7%
дубли: убрано 180, -19.6%
TEST: убрано 3, -2.3%
если убрать только TEST: -8.2%


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [5]:
miss = df[df["delivery_days"].isna()]
print("пропусков:", len(miss))
print()
print(miss["city"].value_counts())
print()
print(miss["status"].value_counts())
print()
print((miss["city"].value_counts() / df["city"].value_counts() * 100).round(1))

пропусков: 158

city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64

status
оплачен    144
отменён     14
Name: count, dtype: int64

city
Актобе       1.0
Алматы       3.0
Астана       3.0
Атырау       3.5
Караганда    4.6
Шымкент      2.8
Name: count, dtype: float64


In [6]:
sh = df.loc[df["city"] == "Шымкент", "delivery_days"]
print("пропусков в Шымкенте:", sh.isna().sum(), "из", len(sh))
print("медиана:", sh.median())

a = (sh.dropna() <= 2).mean() * 100
b = (sh.fillna(sh.median()) <= 2).mean() * 100
c = (sh.fillna(0) <= 2).mean() * 100
print(f"(а) dropna: {a:.1f}%")
print(f"(б) fillna(медиана): {b:.1f}%")
print(f"(в) fillna(0): {c:.1f}%")

пропусков в Шымкенте: 22 из 776
медиана: 5.0
(а) dropna: 29.8%
(б) fillna(медиана): 29.0%
(в) fillna(0): 31.8%


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [7]:
bad = pd.read_csv("orders.csv")
bad.drop_duplicates()
bad["city"] = bad["city"].str.lower()
print(bad["city"].nunique(), len(bad))
print(sorted(bad["city"].unique()))

10 5463
[' алматы', 'almaty', 'astana ', 'shymkent', 'актобе', 'алматы', 'астана', 'атырау', 'караганда', 'шымкент']


In [8]:
fixed = pd.read_csv("orders.csv")
fixed = fixed.drop_duplicates()
fixed["city"] = fixed["city"].str.strip().str.lower().map(CITY)
print(fixed["city"].nunique(), len(fixed))

6 5283


df.drop_duplicates() возвращает новую таблицу, а нам нужно поменять старую поэтому нужно df = df.drop_duplicates()

в названии городов есть еще пробелы и на латинице, поэтому нужно str.strip() и перевод через словарь CITY

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [9]:
week = df["order_date"].dt.day_name().value_counts()
print(week)
print()
low = week.idxmin()
high = week.idxmax()
print(f"меньше всего: {low} ({week.min()}), больше всего: {high} ({week.max()})")
print(f"разница: {(week.max() - week.min()) / week.max() * 100:.1f}%")

order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64

меньше всего: Monday (684), больше всего: Thursday (786)
разница: 13.0%


## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [10]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [11]:
# 8